In [1]:
import pandas as pd
import numpy as np

In [2]:
sell_price = pd.read_csv('dataset/sell_prices.csv')
calendar = pd.read_csv('dataset/calendar.csv')
train = pd.read_csv('dataset/sales_train_evaluation.csv')


In [9]:
sell_price.head()

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


In [11]:
train.head()

,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,d_5,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,...,2,4,0,0,0,0,3,3,0,1
1,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,...,0,1,2,1,1,0,0,0,0,0
2,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,...,1,0,2,0,0,0,2,3,0,1
3,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,...,1,1,0,4,0,1,3,0,2,6
4,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,...,0,0,0,2,1,0,0,2,1,0


In [10]:
# Get column groups
cat_cols = ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
ts_cols = [col for col in train.columns if col not in cat_cols]
ts_dict = {t: int(t[2:]) for t in ts_cols}

# Describe data
print('  unique forecasts: %i' % train.shape[0])
for col in cat_cols:
    print('   N_unique %s: %i' % (col, train[col].nunique()))

  unique forecasts: 30490
   N_unique item_id: 3049
   N_unique dept_id: 7
   N_unique cat_id: 3
   N_unique store_id: 10
   N_unique state_id: 3


In [12]:
# 1. All products, all stores, all states (1 series)
all_sales = pd.DataFrame(train[ts_cols].sum()).transpose()
all_sales['id_str'] = 'all'
all_sales = all_sales[ ['id_str'] +  [c for c in all_sales if c not in ['id_str']] ]
all_sales

,id_str,d_1,d_2,d_3,d_4,d_5,d_6,d_7,d_8,d_9,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,all,32631,31749,23783,25412,19146,29211,28010,37932,32736,...,44073,54308,59921,42362,38777,37096,36963,42552,51518,54338


In [13]:
# 2. All products by state (3 series)
state_sales = train.groupby('state_id',as_index=False)[ts_cols].sum()
state_sales['id_str'] = state_sales['state_id'] 
state_sales = state_sales[ ['id_str'] +  [c for c in state_sales if c not in ['id_str']] ]
state_sales = state_sales.drop(['state_id'],axis=1)
state_sales

,id_str,d_1,d_2,d_3,d_4,d_5,d_6,d_7,d_8,d_9,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,CA,14195,13805,10108,11047,9925,11322,12251,16610,14696,...,18471,23037,24704,17721,16150,15678,16297,17430,23103,24644
1,TX,9438,9630,6778,7381,5912,9006,6226,9440,9376,...,12320,12366,16249,12228,11370,10375,9162,12303,13681,14815
2,WI,8998,8314,6897,6984,3309,8883,9533,11882,8664,...,13282,18905,18968,12413,11257,11043,11504,12819,14734,14879


In [14]:
# 3. All products by store (10 series)
store_sales = train.groupby('store_id',as_index=False)[ts_cols].sum()
store_sales['id_str'] = store_sales['store_id'] 
store_sales = store_sales[ ['id_str'] +  [c for c in store_sales if c not in ['id_str']] ]
store_sales = store_sales.drop(['store_id'],axis=1)
store_sales

,id_str,d_1,d_2,d_3,d_4,d_5,d_6,d_7,d_8,d_9,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,CA_1,4337,4155,2816,3051,2630,3276,3450,5437,4340,...,4951,6245,6707,4568,3949,3995,4136,4433,5764,6289
1,CA_2,3494,3046,2121,2324,1942,2288,2629,3729,2957,...,4839,6507,6710,4238,3963,3789,4037,4751,7120,6614
2,CA_3,4739,4827,3785,4232,3817,4369,4703,5456,5581,...,6046,7029,7969,5891,5523,5375,5580,5542,7073,8144
3,CA_4,1625,1777,1386,1440,1536,1389,1469,1988,1818,...,2635,3256,3318,3024,2715,2519,2544,2704,3146,3597
4,TX_1,2556,2687,1822,2258,1694,2734,1691,2820,2887,...,3610,3787,5000,3708,3310,3147,2748,3664,4167,4624
5,TX_2,3852,3937,2731,2954,2492,3439,2588,3772,3657,...,4415,4424,5866,4179,4008,3518,3126,4249,4802,5217
6,TX_3,3030,3006,2225,2169,1726,2833,1947,2848,2832,...,4295,4155,5383,4341,4052,3710,3288,4390,4712,4974
7,WI_1,2704,2194,1562,1251,2,2049,2815,3248,1674,...,3978,5527,5488,3505,3323,3242,3478,3813,5002,5054
8,WI_2,2256,1922,2018,2522,1175,2244,2232,2643,2140,...,5317,7704,7586,5011,4628,4533,4628,4880,5213,5068
9,WI_3,4038,4198,3317,3211,2132,4590,4486,5991,4850,...,3987,5674,5894,3897,3306,3268,3398,4126,4519,4757


In [15]:
# 4. All products by category (3 series)
cat_sales = train.groupby('cat_id',as_index=False)[ts_cols].sum()
cat_sales['id_str'] = cat_sales['cat_id'] 
cat_sales = cat_sales[ ['id_str'] +  [c for c in cat_sales if c not in ['id_str']] ]
cat_sales = cat_sales.drop(['cat_id'],axis=1)
cat_sales

,id_str,d_1,d_2,d_3,d_4,d_5,d_6,d_7,d_8,d_9,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,FOODS,23178,22758,17174,18878,14603,22093,20490,27751,24862,...,30107,36576,41447,29158,26148,24790,24737,28136,33599,35967
1,HOBBIES,3764,3357,2682,2669,1814,3220,2944,3986,2899,...,3871,4893,5016,3528,3629,3740,3475,4143,5333,5280
2,HOUSEHOLD,5689,5634,3927,3865,2729,3898,4576,6195,4975,...,10095,12839,13458,9676,9000,8566,8751,10273,12586,13091


In [16]:
# 5. All products by department (7 series)
dept_sales = train.groupby('dept_id',as_index=False)[ts_cols].sum()
dept_sales['id_str'] = dept_sales['dept_id'] 
dept_sales = dept_sales[ ['id_str'] +  [c for c in dept_sales if c not in ['id_str']] ]
dept_sales = dept_sales.drop(['dept_id'],axis=1)
dept_sales

,id_str,d_1,d_2,d_3,d_4,d_5,d_6,d_7,d_8,d_9,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,FOODS_1,2343,2216,1657,1508,1209,1897,1903,2235,1925,...,4130,4257,4225,3264,3119,3073,3251,3735,4195,4031
1,FOODS_2,4094,4209,3174,3606,2869,4375,3349,4384,4273,...,5842,7563,9069,6226,5311,4648,4551,4815,5940,6535
2,FOODS_3,16741,16333,12343,13764,10525,15821,15238,21132,18664,...,20135,24756,28153,19668,17718,17069,16935,19586,23464,25401
3,HOBBIES_1,3610,3172,2497,2531,1714,3133,2855,3831,2741,...,3422,4414,4464,3109,3142,3291,3027,3761,4832,4720
4,HOBBIES_2,154,185,185,138,100,87,89,155,158,...,449,479,552,419,487,449,448,382,501,560
5,HOUSEHOLD_1,4105,3858,2827,2732,1802,2664,3161,4334,3494,...,7936,10042,10572,7609,6999,6641,6759,7974,9668,10165
6,HOUSEHOLD_2,1584,1776,1100,1133,927,1234,1415,1861,1481,...,2159,2797,2886,2067,2001,1925,1992,2299,2918,2926


In [4]:
# 转成长格式
sales_long = train.melt(
    id_vars=["item_id","dept_id","cat_id","store_id","state_id"],
    var_name="d",
    value_name="sales"
)

In [20]:
sales_long.head(10)

,item_id,dept_id,cat_id,store_id,state_id,d,sales
0,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
1,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
2,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
3,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
4,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
5,HOBBIES_1_006,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
6,HOBBIES_1_007,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
7,HOBBIES_1_008,HOBBIES_1,HOBBIES,CA_1,CA,d_1,12
8,HOBBIES_1_009,HOBBIES_1,HOBBIES,CA_1,CA,d_1,2
9,HOBBIES_1_010,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0


In [21]:
sales_long.groupby("item_id")["sales"].describe()

,count,mean,std,min,25%,50%,75%,max
item_id,,,,,,,,
FOODS_1_001,19410.0,0.639052,1.315797,0.0,0.0,0.0,1.0,24.0
FOODS_1_002,19410.0,0.384596,0.761401,0.0,0.0,0.0,1.0,8.0
FOODS_1_003,19410.0,0.699536,1.256232,0.0,0.0,0.0,1.0,22.0
FOODS_1_004,19410.0,6.813550,8.421542,0.0,0.0,4.0,11.0,116.0
FOODS_1_005,19410.0,1.206543,2.249736,0.0,0.0,0.0,2.0,30.0
...,...,...,...,...,...,...,...,...
HOUSEHOLD_2_512,19410.0,0.603143,1.111164,0.0,0.0,0.0,1.0,17.0
HOUSEHOLD_2_513,19410.0,0.412777,1.100467,0.0,0.0,0.0,0.0,20.0
HOUSEHOLD_2_514,19410.0,0.164812,0.449395,0.0,0.0,0.0,0.0,5.0


类型	条件	含义
smooth	ADI < 1.32 且 CV² < 0.49	稳定需求
erratic	ADI < 1.32 且 CV² ≥ 0.49	波动大
intermittent	ADI ≥ 1.32 且 CV² < 0.49	偶尔卖
lumpy	ADI ≥ 1.32 且 CV² ≥ 0.49	很不规律

cv2 = (std / mean) ** 2
ADI = 平均非零间隔
ADI = 总天数 / 非零天数

In [22]:
def compute_adi(x):
    non_zero = (x > 0).sum()
    return len(x) / non_zero if non_zero > 0 else np.nan

def compute_cv2(x):
    mean = x.mean()
    std = x.std()
    return (std / mean) ** 2 if mean > 0 else np.nan

In [24]:
stats = sales_long.groupby("item_id")["sales"].agg([
    ("mean", "mean"),
    ("std", "std"),
    ("ADI", compute_adi),
    ("CV2", compute_cv2)
]).reset_index()

In [25]:
stats.head()

,item_id,mean,std,ADI,CV2
0,FOODS_1_001,0.639052,1.315797,2.867484,4.239406
1,FOODS_1_002,0.384596,0.761401,3.726959,3.919384
2,FOODS_1_003,0.699536,1.256232,2.669876,3.224920
3,FOODS_1_004,6.813550,8.421542,1.524625,1.527694
4,FOODS_1_005,1.206543,2.249736,2.209196,3.476781


In [26]:
def classify(row):
    if row["ADI"] < 1.32 and row["CV2"] < 0.49:
        return "smooth"
    elif row["ADI"] < 1.32 and row["CV2"] >= 0.49:
        return "erratic"
    elif row["ADI"] >= 1.32 and row["CV2"] < 0.49:
        return "intermittent"
    else:
        return "lumpy"

stats["type"] = stats.apply(classify, axis=1)

In [27]:
stats.head()

,item_id,mean,std,ADI,CV2,type
0,FOODS_1_001,0.639052,1.315797,2.867484,4.239406,lumpy
1,FOODS_1_002,0.384596,0.761401,3.726959,3.919384,lumpy
2,FOODS_1_003,0.699536,1.256232,2.669876,3.224920,lumpy
3,FOODS_1_004,6.813550,8.421542,1.524625,1.527694,lumpy
4,FOODS_1_005,1.206543,2.249736,2.209196,3.476781,lumpy


In [30]:
stats['type'].value_counts()

type
lumpy      2961
erratic      82
smooth        6
Name: count, dtype: int64

In [31]:
stats["CV2"].describe()

count    3049.000000
mean        6.343465
std         5.959186
min         0.313519
25%         2.502051
50%         4.276849
75%         8.013566
max        71.250721
Name: CV2, dtype: float64

type
lumpy      2961
erratic      82
smooth        6
Name: count, dtype: int64 
数据不平衡 90%是0 模型可能直接预测0
Due to strong demand heterogeneity (e.g., lumpy vs smooth demand), a single global forecasting model may suffer from bias toward dominant patterns (e.g., zero-heavy series), leading to suboptimal performance.
直接说明了分需求预测的合理性
A global baseline model (trained on all items)
A segmented modeling approach (trained separately by demand type)

简单平均模型 不考虑季节、趋势、门店差异。
全局平均：每个商品未来销量 = 历史销量平均。

In [5]:
# 按 item_id + store_id 计算历史平均销量
baseline = sales_long.groupby(['item_id','store_id'])['sales'].mean().reset_index()
baseline.rename(columns={'sales':'pred'}, inplace=True)

# 生成未来 28 天的预测
future_days = [f'd_{i}' for i in range(1914, 1942)]
for d in future_days:
    baseline[d] = baseline['pred']

移动平均或最后 N 天平均
用最近 28 天销量平均，预测未来 28 天。
更能反映近期趋势。

In [3]:
# 最近 28 天列名
recent_days = [f'd_{i}' for i in range(1913-28+1, 1913+1)]  # d_1886 ~ d_1913

# 每行商品+门店最近 28 天平均销量
baseline_values = train[recent_days].mean(axis=1)

# 生成未来 28 天预测列
future_days = [f'd_{i}' for i in range(1914, 1942)]
import numpy as np
baseline = pd.DataFrame(
    np.tile(baseline_values.values.reshape(-1,1), len(future_days)),
    columns=future_days
)


In [4]:
baseline

,d_1914,d_1915,d_1916,d_1917,d_1918,d_1919,d_1920,d_1921,d_1922,d_1923,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,0.964286,0.964286,0.964286,0.964286,0.964286,0.964286,0.964286,0.964286,0.964286,0.964286,...,0.964286,0.964286,0.964286,0.964286,0.964286,0.964286,0.964286,0.964286,0.964286,0.964286
1,0.071429,0.071429,0.071429,0.071429,0.071429,0.071429,0.071429,0.071429,0.071429,0.071429,...,0.071429,0.071429,0.071429,0.071429,0.071429,0.071429,0.071429,0.071429,0.071429,0.071429
2,0.571429,0.571429,0.571429,0.571429,0.571429,0.571429,0.571429,0.571429,0.571429,0.571429,...,0.571429,0.571429,0.571429,0.571429,0.571429,0.571429,0.571429,0.571429,0.571429,0.571429
3,1.821429,1.821429,1.821429,1.821429,1.821429,1.821429,1.821429,1.821429,1.821429,1.821429,...,1.821429,1.821429,1.821429,1.821429,1.821429,1.821429,1.821429,1.821429,1.821429,1.821429
4,1.357143,1.357143,1.357143,1.357143,1.357143,1.357143,1.357143,1.357143,1.357143,1.357143,...,1.357143,1.357143,1.357143,1.357143,1.357143,1.357143,1.357143,1.357143,1.357143,1.357143
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30485,0.214286,0.214286,0.214286,0.214286,0.214286,0.214286,0.214286,0.214286,0.214286,0.214286,...,0.214286,0.214286,0.214286,0.214286,0.214286,0.214286,0.214286,0.214286,0.214286,0.214286
30486,0.321429,0.321429,0.321429,0.321429,0.321429,0.321429,0.321429,0.321429,0.321429,0.321429,...,0.321429,0.321429,0.321429,0.321429,0.321429,0.321429,0.321429,0.321429,0.321429,0.321429
30487,0.892857,0.892857,0.892857,0.892857,0.892857,0.892857,0.892857,0.892857,0.892857,0.892857,...,0.892857,0.892857,0.892857,0.892857,0.892857,0.892857,0.892857,0.892857,0.892857,0.892857
30488,0.928571,0.928571,0.928571,0.928571,0.928571,0.928571,0.928571,0.928571,0.928571,0.928571,...,0.928571,0.928571,0.928571,0.928571,0.928571,0.928571,0.928571,0.928571,0.928571,0.928571
